# Melanoma metadata cleaning

Clean the train and test metadata consistently while preserving every row and leaving the raw files unchanged.

- Missing `sex`, `age_approx`, and `anatom_site_general_challenge` values become `unknown`.
- Because `age_approx` is supplied in discrete five-year buckets, it is retained as a categorical string. Missing age is **not** inferred from other patients.
- Identifiers and labels are validated, but `diagnosis` and `benign_malignant` must not be model features because they reveal the outcome and are absent from the test data.
- New files are written to `data/processed`


## Setup

Set `DATA_DIR` below to the local folder containing the `raw` and `processed` directories.


In [ ]:
from pathlib import Path

import pandas as pd

# Change this one path to your local data directory.
DATA_DIR = Path(r"C:/Users/thoma/Documents/GitHub/ADS2002-Melanoma/data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

TRAIN_PATH = RAW_DIR / "train.csv"
TEST_PATH = RAW_DIR / "test.csv"
TRAIN_CLEAN_PATH = PROCESSED_DIR / "train_clean.csv"
TEST_CLEAN_PATH = PROCESSED_DIR / "test_clean.csv"

UNKNOWN_LABEL = "unknown"
MODEL_FEATURES = ["sex", "age_approx", "anatom_site_general_challenge"]
LEAKAGE_COLUMNS = ["diagnosis", "benign_malignant", "target"]

print(f"Data directory: {DATA_DIR}")


Data directory: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data


## Data

### 1. Load raw metadata and report its shape


In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print(f"Training rows: {len(train_raw):,}; columns: {train_raw.shape[1]}")
print(f"Test rows: {len(test_raw):,}; columns: {test_raw.shape[1]}")


Training rows: 33,126; columns: 8
Test rows: 10,982; columns: 5


### 2. Profile missing values before cleaning


In [3]:
def missing_profile(frame: pd.DataFrame) -> pd.DataFrame:
    missing_count = frame.isna().sum()
    return pd.DataFrame(
        {
            "missing_count": missing_count,
            "missing_percent": (100 * missing_count / len(frame)).round(3),
        }
    ).query("missing_count > 0")


print("Training missingness:")
print(missing_profile(train_raw).to_string())
print("\nTest missingness:")
print(missing_profile(test_raw).to_string())


Training missingness:
                               missing_count  missing_percent
sex                                       65            0.196
age_approx                                68            0.205
anatom_site_general_challenge            527            1.591

Test missingness:
                               missing_count  missing_percent
anatom_site_general_challenge            351            3.196


## Cleaning

Cleaning is deterministic: no statistics are learned from the full dataset, so applying it before cross-validation does not leak validation information.


In [4]:
EXPECTED_SEX = {"female", "male", UNKNOWN_LABEL}
EXPECTED_SITES = {
    "head/neck",
    "lower extremity",
    "oral/genital",
    "palms/soles",
    "torso",
    "upper extremity",
    UNKNOWN_LABEL,
}


def strip_text_columns(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in cleaned.select_dtypes(include=["object", "string"]).columns:
        values = cleaned[column].astype("string").str.strip()
        cleaned[column] = values.mask(values.eq(""), pd.NA)
    return cleaned


def clean_metadata(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = strip_text_columns(frame)

    required_columns = {
        "image_name",
        "patient_id",
        "sex",
        "age_approx",
        "anatom_site_general_challenge",
    }
    missing_columns = required_columns - set(cleaned.columns)
    if missing_columns:
        raise ValueError(f"Required columns are missing: {sorted(missing_columns)}")

    for column in ["sex", "anatom_site_general_challenge"]:
        cleaned[column] = cleaned[column].fillna(UNKNOWN_LABEL)

    age_numeric = pd.to_numeric(cleaned["age_approx"], errors="coerce")
    malformed_age = cleaned["age_approx"].notna() & age_numeric.isna()
    if malformed_age.any():
        bad_values = sorted(cleaned.loc[malformed_age, "age_approx"].unique())
        raise ValueError(f"Non-numeric age values found: {bad_values}")

    known_age = age_numeric.dropna()
    if not known_age.between(0, 120).all():
        raise ValueError("age_approx contains values outside the expected 0-120 range.")
    if not known_age.mod(5).eq(0).all():
        raise ValueError("age_approx contains values outside the expected five-year buckets.")

    cleaned["age_approx"] = age_numeric.map(
        lambda value: str(int(value)) if pd.notna(value) else UNKNOWN_LABEL
    )

    if "target" in cleaned.columns:
        cleaned["target"] = pd.to_numeric(cleaned["target"], errors="raise").astype("int8")

    return cleaned


train_clean = clean_metadata(train_raw)
test_clean = clean_metadata(test_raw)


## Checks

### 3. Validate cleaned datasets


In [5]:
def validate_common(frame: pd.DataFrame, source: pd.DataFrame, name: str) -> None:
    assert len(frame) == len(source), f"{name}: row count changed during cleaning."
    assert frame["image_name"].notna().all(), f"{name}: missing image_name."
    assert frame["patient_id"].notna().all(), f"{name}: missing patient_id."
    assert frame["image_name"].is_unique, f"{name}: duplicate image_name values."
    assert set(frame["sex"].unique()) <= EXPECTED_SEX, f"{name}: unexpected sex value."
    assert set(frame["anatom_site_general_challenge"].unique()) <= EXPECTED_SITES, (
        f"{name}: unexpected anatomical-site value."
    )
    assert frame[MODEL_FEATURES].isna().sum().sum() == 0, (
        f"{name}: model features still contain missing values."
    )


validate_common(train_clean, train_raw, "train")
validate_common(test_clean, test_raw, "test")

assert set(train_clean["target"].unique()) <= {0, 1}, "Unexpected target value."
assert (
    ((train_clean["target"] == 1) & (train_clean["benign_malignant"] != "malignant"))
    | ((train_clean["target"] == 0) & (train_clean["benign_malignant"] != "benign"))
).sum() == 0, "target and benign_malignant disagree."

print("All validation checks passed.")


All validation checks passed.


### 4. Reconcile the transformations


In [6]:
rows = []
for dataset_name, source, cleaned in [
    ("train", train_raw, train_clean),
    ("test", test_raw, test_clean),
]:
    for column in MODEL_FEATURES:
        rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "missing_before": int(source[column].isna().sum()),
                "unknown_after": int(cleaned[column].eq(UNKNOWN_LABEL).sum()),
                "missing_after": int(cleaned[column].isna().sum()),
            }
        )

cleaning_summary = pd.DataFrame(rows)
print(cleaning_summary.to_string(index=False))


dataset                        column  missing_before  unknown_after  missing_after
  train                           sex              65             65              0
  train                    age_approx              68             68              0
  train anatom_site_general_challenge             527            527              0
   test                           sex               0              0              0
   test                    age_approx               0              0              0
   test anatom_site_general_challenge             351            351              0


## Results

### 5. Save new processed files


In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_clean.to_csv(TRAIN_CLEAN_PATH, index=False)
test_clean.to_csv(TEST_CLEAN_PATH, index=False)

for path in [TRAIN_CLEAN_PATH, TEST_CLEAN_PATH]:
    print(f"Wrote: {path}")


Wrote: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\processed\train_clean.csv
Wrote: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\processed\test_clean.csv
